In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew
import igraph
import os

pd.set_option("display.max_columns", 30)
ddir = "/home/scai/PhenPred/data/"
data_folder = "/home/scai/PhenPred/data/clines"
# Import samplesheets
cols = ["model_id", "BROAD_ID", "tissue", "cancer_type"]
col_rename = dict(
    ModelID="BROAD_ID",
    SangerModelID="model_id",
    SampleCollectionSite="tissue",
    OncotreeLineage="cancer_type",
)
ss_cmp = pd.read_csv(f"{data_folder}/model_list_20230505.csv")

ss_depmap = pd.read_csv(f"{data_folder}/depmap24Q4/Model.csv")
ss_depmap.rename(columns=col_rename, inplace=True)

# Map sample IDs to Sanger IDs
samplesheet = pd.concat(
    [
        ss_cmp[cols].dropna().assign(source="sanger"),
        ss_depmap[cols].dropna().assign(source="broad"),
    ]
)
samplesheet = samplesheet.groupby("model_id").first().reset_index()
samplesheet.replace(
    {
        "tissue": dict(
            large_intestine="Large Intestine",
            lung="Lung",
            ovary="Ovary",
            haematopoietic_and_lymphoid_tissue="Haematopoietic and Lymphoid",
            bone_marrow="Other tissue",
            upper_aerodigestive_tract="Other tissue",
            ascites="Other tissue",
            pleural_effusion="Other tissue",
        )
    },
    inplace=True,
)
tissue_map = samplesheet.set_index("model_id").to_dict()["tissue"]

# Growth
growth = pd.read_csv(f"{data_folder}/growth_rate_20220907.csv")
growth = (
    growth.sort_values(["model_id", "replicates"], ascending=False)
    .groupby("model_id")
    .first()
)
growth = growth.dropna(subset=["day4_day1_ratio"])

In [2]:
timestamp = "20250508_160635"
## Transcriptomics
gexp_df = pd.read_csv(
    f"/home/scai/PhenPred/reports/vae/files/{timestamp}_imputed_transcriptomics.csv.gz",
    index_col=0,
)

## CRISPR-Cas9
cas9_df = pd.read_csv(
    f"/home/scai/PhenPred/reports/vae/files/{timestamp}_imputed_crisprcas9.csv.gz",
    index_col=0,
)

In [3]:
df_res_vae_annot = pd.read_csv(
    f"../reports/vae/crispr/{timestamp}_transcriptomics_crisprcas9_remove_latent_n3_no_tissue_standardizedTrue_annot.csv.gz"
)

In [16]:
df_res_vae_annot["log10fdr_orig"] = -np.log10(df_res_vae_annot["fdr_orig"])
df_res_vae_annot["log10fdr_vae"] = -np.log10(df_res_vae_annot["fdr_vae"])
df_res_vae_annot["diff_log10fdr"] = (
    df_res_vae_annot["log10fdr_vae"] - df_res_vae_annot["log10fdr_orig"]
)

In [17]:
# df_res_vae_filtered = df_res_vae_annot[df_res_vae_annot["beta_vae"] > 0]
df_res_vae_filtered = df_res_vae_annot.query("fdr_vae < 1e-2")
df_res_vae_filtered = df_res_vae_filtered.query("entropy > 0.5")

In [12]:
df_res_vae_filtered

,y_id,x_id,n_orig,beta_orig,lr_orig,covs_orig,pval_orig,fdr_orig,n_vae,beta_vae,lr_vae,covs_vae,pval_vae,fdr_vae,skew_orig,skew_mosa,target_detailed,target,entropy
0,FAM50A,FAM50B,923.0,0.75825,775.49438,208.0,1.148490e-170,1.975632e-166,1523.0,0.62305,846.16667,211.0,4.953857e-186,1.802213e-182,-0.62729,-0.38142,No link; CRISPR not in network,-,0.82245
1,DDX3X,DDX3Y,923.0,0.70104,628.13085,208.0,1.274517e-138,2.192424e-134,1523.0,0.54253,592.67442,211.0,6.564286e-131,2.388087e-127,0.54274,-0.18181,3,3,0.83160
2,DDX3X,USP9Y,923.0,0.66805,569.93911,208.0,5.788442e-126,9.957278e-122,1523.0,0.50986,522.08826,211.0,1.486670e-115,5.408505e-112,0.54274,-0.18181,3,3,0.83160
3,DDX3X,UTY,923.0,0.67039,566.34511,208.0,3.502334e-125,6.024714e-121,1523.0,0.50949,523.46762,211.0,7.449375e-116,2.710083e-112,0.54274,-0.18181,3,3,0.83160
4,EIF1AX,EIF1AY,923.0,0.69643,563.47514,208.0,1.474560e-124,2.536539e-120,1523.0,0.54443,528.78237,211.0,5.198192e-117,1.891102e-113,0.61340,-0.24279,1,1,0.83172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4726410,TP63,KRT15,923.0,-0.07417,2.78142,208.0,9.536366e-02,6.223515e-01,1523.0,-0.16477,23.26264,211.0,1.413186e-06,5.141169e-03,-2.40368,-2.15634,No link; Gene not in network,-,0.55644
5096815,CDS2,C1orf210,923.0,0.11235,3.15583,208.0,7.565628e-02,6.370414e-01,1523.0,0.21988,26.06644,211.0,3.298676e-07,1.200059e-03,-1.00569,-0.35216,No link; Gene not in network,-,0.72541
5575406,UQCRFS1,SYDE1,923.0,0.08637,2.62630,208.0,1.051064e-01,6.548687e-01,1523.0,0.17636,17.20674,211.0,3.352441e-05,7.622613e-03,-0.18528,0.13470,No link; Gene not in network,-,0.82794
7275419,AP2M1,GJA1,923.0,0.05398,1.89575,208.0,1.685550e-01,7.089456e-01,1523.0,0.10985,14.88335,211.0,1.143678e-04,9.241912e-03,-0.25145,-0.36897,3,3,0.82125


In [26]:
COLS = [
    "y_id",
    "x_id",
    "beta_orig",
    "beta_vae",
    "fdr_orig",
    "fdr_vae",
    "diff_log10fdr",
    "skew_orig",
    "skew_mosa",
    "target",
    "entropy",
]
df_final_formatted = df_res_vae_filtered[COLS].sort_values("fdr_vae", ascending=True)
# Rename columns from "_vae" to "_mosa" using a more succinct approach
df_final_formatted = df_final_formatted.rename(
    columns={
        col: col.replace("_vae", "_mosa")
        for col in df_final_formatted.columns
        if "_vae" in col
    }
)
# Rename y_id to crispr_gene and x_id to rna_gene
df_final_formatted = df_final_formatted.rename(
    columns={"y_id": "crispr_gene", "x_id": "rna_gene"}
)

In [27]:
df_final_formatted

,crispr_gene,rna_gene,beta_orig,beta_mosa,fdr_orig,fdr_mosa,diff_log10fdr,skew_orig,skew_mosa,target,entropy
0,FAM50A,FAM50B,0.75825,0.62305,1.975632e-166,1.802213e-182,16.039900,-0.62729,-0.38142,-,0.82245
1,DDX3X,DDX3Y,0.70104,0.54253,2.192424e-134,2.388087e-127,-7.037126,0.54274,-0.18181,3,0.83160
4,EIF1AX,EIF1AY,0.69643,0.54443,2.536539e-120,1.891102e-113,-6.872473,0.61340,-0.24279,1,0.83172
3,DDX3X,UTY,0.67039,0.50949,6.024714e-121,2.710083e-112,-8.653046,0.54274,-0.18181,3,0.83160
2,DDX3X,USP9Y,0.66805,0.50986,9.957278e-122,5.408505e-112,-9.734937,0.54274,-0.18181,3,0.83160
...,...,...,...,...,...,...,...,...,...,...,...
48042,PPA2,SH3TC2,-0.17547,-0.16031,9.981282e-02,9.997183e-03,0.999309,-0.71976,-0.33971,-,0.77197
73848,BRAT1,SH3TC2,0.19997,0.18490,1.245245e-01,9.997183e-03,1.095377,-0.45364,0.17730,-,0.84801
57608,THAP3,LMNA,0.20884,0.21612,1.099670e-01,9.999936e-03,1.041265,-1.08569,-0.83347,-,0.60561
156795,AP1G1,ADGRL3,0.13563,0.13020,1.770708e-01,9.999956e-03,1.248149,-0.76317,-1.01803,3,0.57375


In [28]:
df_final_formatted.to_excel(
    f"../reports/vae/crispr/crisprcas9_transcriptomics_SL_pairs_Dave_{timestamp}_popcorrected.xlsx",
    index=False,
)

In [35]:
mcc_melt_df = pd.read_csv(f"../reports/vae/crispr/{timestamp}_mcc_results.csv")

In [36]:
mcc_melt_df

,CRISPR:GEXP,crispr_gene,rna_gene,Mutation,mcc
0,WRN:KRT6C,WRN,KRT6C,RPL22,0.58542
1,WRN:KRT6C,WRN,KRT6C,ACVR2A,0.53221
2,WRN:KRT6C,WRN,KRT6C,BMPR2,0.52365
3,WRN:GPR176,WRN,GPR176,BMPR2,0.49406
4,WRN:GPR176,WRN,GPR176,ACVR2A,0.47483
...,...,...,...,...,...
56012,NAA10:PAGE5,NAA10,PAGE5,BRAF,-0.22688
56013,WDR77:CDKN2A,WDR77,CDKN2A,TP53,-0.24953
56014,SNAPC5:CYP2W1,SNAPC5,CYP2W1,APC,-0.25899
56015,NUP93:IL13RA2,NUP93,IL13RA2,BRAF,-0.29063


In [37]:
mcc_melt_df_filtered = mcc_melt_df.query("mcc > 0.2").sort_values(
    "mcc", ascending=False
)

In [40]:
df_final_mcc_filtered = pd.merge(
    df_final_formatted,
    mcc_melt_df_filtered,
    on=["crispr_gene", "rna_gene"],
    how="inner",
)

In [42]:
df_final_mcc_filtered.to_excel(
    f"../reports/vae/crispr/SL_pairs_Dave_{timestamp}_mcc_filtered.xlsx",
    index=False,
)
